In [1]:
#import keras
#from keras import layers
#from keras.layers import TextVectorization
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.layers import TextVectorization

import tensorflow as tf

import matplotlib.pyplot as plt
import numpy as np

from ipywidgets import interact

### Methoden en Technieken (M&T), blok 2 - Les 4

In de vorige les hebben we de Transformer geintroduceerd.

<br>

In deze les introduceren we verschillende toepassingen van de Transformer en generatieve tekstmodellen.

<br>

De inhoud is grotendeels gebaseerd op H15 en H16 uit [*Deep Learning with Python*](https://deeplearningwithpython.io/).<br>
De code komt grotendeels van<br>
[https://github.com/fchollet/deep-learning-with-python-notebooks](https://github.com/fchollet/deep-learning-with-python-notebooks)

### Chollet [DLP] over de Transformer

*"After seeing all of these implementation details, one might reasonably protest that this all seems quite arbitrary. There are so many small details to take on faith. How could we possibly know this choice and configuration of layers is optimal?"*

<br>

*"But it’s worth remembering that, as a whole, the field of machine learning moves empirically. Attention grew out of an attempt to augment RNNs, and after years of guessing and checking by a ton of people, it gave rise to the Transformer. There is little reason to think this process is done playing out."*

### Voordeel van Transformer

* Transformer trainde sneller dan de modellen die het verving
* *Data hungry*. Het model blijft langer verbeteren dan de modellen die het verving

<br>

Bovenstaande heeft ervoor gezorgd dat men grotere modellen met meer data is gaan trainen.

* Eerste Transformer is getraind met miljoenen tokens
* Later gebruikte men miljarden tokens
* Tegenwoordig biljoenen ($10^{12}$)

### Welke datasets zijn daarvoor gebruikt?

Wordt steeds onduidelijker voor het publiek

<br>

* Eerste Transformer: Engels-Duitse vertaal-dataset met 4 miljoen zinsparen
* BERT (2018): Engelse Wikipedia met 7000 boeken (16GB)
* GPT2 (2019): tekst van miljoenen web-pagina's (van "voldoende" kwaliteit)
* Llama (2025): *"15 trillion tokens of data from publicly available sources"* volgens Meta

### BERT (Bidirectional Encoder Representations from Transformers)

* Zie [https://arxiv.org/abs/1810.04805](https://arxiv.org/abs/1810.04805)
* Transformer-encoder
* 100-300 miljoen parameters
* unsupervised pre-training voor een word embedding
* 15% van een stuk tekst wordt gemaskeerd
* *masked language modeling*
* model voor $P(\texttt{token} | \texttt{tokens in de buurt})$

### pretrained Transformer

* RoBERTa (Robustly Optimized BERT)
* Tien keer meer data gebruikt tov BERT (160GB)
* train-tijd: een dag op 1024 GPU's

Wat hebben we nodig?

* Een zelfde tokenizer
* Een zelfde modelarchitectuur
* De voorgetrainde gewichten

### [DLP] maakt gebruik van [KerasHub](https://keras.io/keras_hub/)

* KerasHub maakt gebruik van tensorflow-text
* Wordt niet meer onderhouden voor Windows (of macOS)
* Kan ingeladen worden via Huggingface

<br>

* Tokenizer: subword, 50 duizend tokens
* Plaatst een woord/token in een word-embedding (768 dimensies)
* Noemen we ook wel *latente ruimte*
* 12 gestapelde Transformer-encoders
* Iets meer dan 124 miljoen gewichten

### Classificeren (binair) met RoBERTa

* RoBERTa:
    * Invoer: tekst
    * Uitvoer: reeks van vectoren (768 dimensies) in de context van de vectoren eromheen
* Om te classificeren voegen we een "kop" toe met
    * Invoer: reeks van vectoren
    * Uitvoer: 1 kans (binaire variabele)
* Hoe vatten we de reeks van vectoren samen?
    * Kies de eerste vector in de reeks
* 1 laag met niet-lineaire activatie


### Gebruik van voorgetraind model

* IMDB-classificatie uit de vorige lessen
    * Op COLAB met GPU: +/- half uur
    * Na 1 epoch 93% accuraat
* Op mijn CPU zou dit meer dan 40 uur duren
    * Heb het model van Huggingface, niet van KerasHub
* Welke keuze maakt Chollet in [DLP]?
    * Alle gewichten doortrainen met kleine Learning rate
    * Voorgetraind model; feature extraction versus fine tuning

### Wat maakt de Transformer zo effectief?

* Word embedding $\texttt{Word2Vec}$; vergelijkbaar met CBOW
* Zorgt ervoor dat woorden die op elkaar lijken in dezelfde richting staan
* Waarom is dat zo?
* Men is erachter gekomen dat in de inbedding richtingen zijn met betekenis
* Attention maakt gebruik van hetzelfde mechanisme
* In deze inbeddingen
    * Zal een punt in de buurt van de vector ongeveer hetzelfde betekenen
    * Zal een punt tussen twee vectoren wat betreft betekenis er ook tussenin zitten<br>
    Ook als dat tussenpunt nooit onderdeel van de data was

### Verschil $\texttt{Word2Vec}$ en Transformer

* Transformer kan vele malen groter (en dieper zijn)
* Daar waar $\texttt{Word2Vec}$ zoiets kan als $\texttt{plural(cat) → cats}$
* Kan een Transformer zoiets als
$$
\begin{aligned}
&\texttt{write_this_in_style_of_shakespeare("...your poem...")}\\
&\texttt{→ "...new poem..."}
\end{aligned}
$$

* Men kan de *latent space* (verborgen ruimte, *embedding space*) interpreteren als de statistische structuur van een taal
* Het kan ons helpen bij het uitrekenen van de kansen (op specifieke woorden en zinnen).

### Korte geschiedenis van het genereren van rijen

- Rond 2002: het genereren van muziek met LSTM's.
- Rond 2013: het genereren van handschrift (tijdreeks van posities van de pen) middels LSTM.
- 2015-2017: het genereren van tekst  en dialogen, muziek en spraak met RNN's
- 2017: the Transformer neemt het over van de RNN's
- 2018: Generative Pretrained Transformer (GPT)

### GPT

* Unsupervised pretraining van een taalmodel
* Unsupervised taak: wat is het volgende woord?
* Transformer-decoder-architectuur
* Zeer veel tekst-data

<br>

* Geen nieuwe ontwikkeling in type model of manier van trainen
* Inzicht: algemeen model getraind op zeer veel data haalt goede resultaten


### GPT-varianten

* GPT-1 (2018): 117 miljoen parameters, 1 miljard tokens
* GPT-2 (2019): 1,5 miljard parameters, meer dan 10 miljard tokens
* GPT-3 (2020): 175 miljard parameters, ongeveer half biljoen tokens ($0{,}5 \, 10^{12}$)

<br>

* GPT-1: Gebruikt als pretrained-model
* Met fine-tuning zeer algemeen bruikbaar
* Latere GPT-versies kunnen aanzienlijk meer

### GPT-2

* Zonder fine-tuning
* *few-shot learning*
* Gegeven een prompt met enkele voorbeelden, vult het model een nieuw voorbeeld correct aan

<div class="alert alert-block alert-warning">
Translate English to French:<br>
sea otter => loutre de mer<br>
peppermint => menthe poivrée<br>
plush giraffe => peluche girafe<br>
cheese =></div>

### GPT-3

* Kan al correct antwoord geven zonder voorbeelden

<div class="alert alert-block alert-warning">
Translate English to French:<br>
cheese =></div>

* Basis voor ChatGPT
* Probleem: niet altijd correct (zonder indicatie)
* Probleem: gevoelig voor verwoording in prompt
* Probleem: kan alleen problemen oplossen waar ook uitvoerig op is getraind

### Structuur van GPT

* Maakt alleen gebruik van de Transformer-decoder
* Vraag-antwoord-paren horen dan als 1 reeks ingevoerd te worden
* Is dus niet bidirectional
* Voordeel: versimpelde pre-training, alle tekst is bruikbaar

### Sampling
Zodra we eenmaal weten wat de kansen zijn voor een combinatie van uitvoervariabelen, dan is het mogelijk om een steekproef te trekken (*sampling*).

![Fig. 12.1 uit [DLP, 2de editie]](../../2025-2026/Slides/Plaatjes/TextGenerationProcess.png)

### Strategie&euml;n voor sampling
- **Greedy sampling**<br>
Kies steeds het woord met de hoogste kans.<br>
Nadeel: herhalingen en voorspelbare zinnen die niet echt lijken
- **Stochastic sampling**<br>
Kies een woord middels een kansproces.
     - uniforme verdeling (elk symbool even veel kans)
     - de verdeling van het model (is de kans op een woord 0,03 kies het dan drie op de honderd keer)
     
  Nadeel: hoe weten we welk kansproces geschikt is voor het probleem?

### Toevoegen van temperatuur
- Teveel willekeur zorgt voor een te willekeurige volgorde van woorden.
- Te weinig willekeur zorgt voor te weinig opties.

Temperatuur $T$: hyperparameter om de hoeveelheid stochasticiteit in het kansproces te beheersen.

Als het model de kans op een symbool (gegeven de voorgaande woorden $X$) inschat met $P(\textrm{symbool} | X)$, dan zal er een nieuwe kans worden uitgerekend middels

$$
P_{\textrm{nieuw}} (\textrm{symbool} | X) = C_{\textrm{norm}} P(\textrm{symbool} | X)^{\frac{1}{T}}
$$

### Toevoegen van temperatuur, het effect
- Hoe lager $T$ hoe kleiner de variatie
- Hoe hoger $T$ hoe groter de variatie


In [2]:
def reweight_distribution(original_distribution, temperature=0.5):
    distribution = np.log(original_distribution) / temperature
    distribution = np.exp(distribution)
    return distribution / np.sum(distribution)

In [3]:
def interactive_plot(temperature):
	x = ["rood","groen","blauw", "geel"]
	h = [0.1,0.2,0.5,0.2]
	y_new = reweight_distribution(h, temperature)
	plt.figure(figsize=(8, 4))
	plt.bar(x, h, alpha=0.5, label="oorspronkelijk")
	plt.bar(x, y_new, alpha=0.5, label="met temperatuur")
	plt.xlabel('Woord')
	plt.ylabel('Kans')
	plt.legend()
	plt.title('Willekeurige verdeling en temperatuur')
	plt.grid(True)
	plt.show()

In [ ]:
interact(interactive_plot, temperature=(0.1, 10, 0.1))
#interactive_plot(1)

interactive(children=(FloatSlider(value=5.0, description='temperature', max=10.0, min=0.1), Output()), _dom_cl…

<function __main__.interactive_plot(temperature)>

### Voorbeeld voor genereren van tekst
We zullen filmrecensies genereren.

We maken gebruik de IMDB-dataset.

- Data-preparatie:
    - Kleine letter, leestekens weg
    - Ook het symbool $''<\texttt{br /}>''$ weg (hoefde niet voor classificatie)
    - Symbolen zijn woorden
    - Maximaal 100 woorden per recensie
    - Maximaal 15 duizend woorden gebruiken

### Voorbeeld voor genereren van tekst, opzet

- Train:
    - Model is een Transformer-decoder.
    - Encoder is niet nodig (er is geen source; we willen genereren).
    - Net zoals in de vorige les trainen we op tekst en dezelfde verschoven tekst.
    - Zo leert het model om het volgende woord te voorspellen.
    - Causal masking
    - 200 epochs
- Prompt:
    - Begin met "This movie"

In [5]:
#| output: false

dataset = keras.utils.text_dataset_from_directory(
    directory="/Users/Ik/backup_2308/2022-2023/AAI/MenT/Chollet/aclImdb", label_mode=None, batch_size=256)
dataset = dataset.map(lambda x: tf.strings.regex_replace(x, "<br />", " "))

Found 50006 files belonging to 1 classes.


In [6]:
sequence_length = 100
vocab_size = 15000
text_vectorization = TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length,
)
text_vectorization.adapt(dataset)

In [7]:
def prepare_lm_dataset(text_batch):
    vectorized_sequences = text_vectorization(text_batch)
    x = vectorized_sequences[:, :-1]
    y = vectorized_sequences[:, 1:]
    return x, y

lm_dataset = dataset.map(prepare_lm_dataset, num_parallel_calls=4)

In [8]:
class PositionalEmbedding(layers.Layer):
    def __init__(self, sequence_length, input_dim, output_dim, **kwargs):
        super().__init__(**kwargs)
        self.token_embeddings = layers.Embedding(
            input_dim=input_dim, output_dim=output_dim)
        self.position_embeddings = layers.Embedding(
            input_dim=sequence_length, output_dim=output_dim)
        self.sequence_length = sequence_length
        self.input_dim = input_dim
        self.output_dim = output_dim

    def call(self, inputs):
        length = tf.shape(inputs)[-1]
        positions = tf.range(start=0, limit=length, delta=1)
        embedded_tokens = self.token_embeddings(inputs)
        embedded_positions = self.position_embeddings(positions)
        return embedded_tokens + embedded_positions

    def compute_mask(self, inputs, mask=None):
        return tf.math.not_equal(inputs, 0)

    def get_config(self):
        config = super(PositionalEmbedding, self).get_config()
        config.update({
            "output_dim": self.output_dim,
            "sequence_length": self.sequence_length,
            "input_dim": self.input_dim,
        })
        return config


class TransformerDecoder(layers.Layer):
    def __init__(self, embed_dim, dense_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.dense_dim = dense_dim
        self.num_heads = num_heads
        self.attention_1 = layers.MultiHeadAttention(
          num_heads=num_heads, key_dim=embed_dim)
        self.attention_2 = layers.MultiHeadAttention(
          num_heads=num_heads, key_dim=embed_dim)
        self.dense_proj = keras.Sequential(
            [layers.Dense(dense_dim, activation="relu"),
             layers.Dense(embed_dim),]
        )
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()
        self.layernorm_3 = layers.LayerNormalization()
        self.supports_masking = True

    def get_config(self):
        config = super(TransformerDecoder, self).get_config()
        config.update({
            "embed_dim": self.embed_dim,
            "num_heads": self.num_heads,
            "dense_dim": self.dense_dim,
        })
        return config

    def get_causal_attention_mask(self, inputs):
        input_shape = tf.shape(inputs)
        batch_size, sequence_length = input_shape[0], input_shape[1]
        i = tf.range(sequence_length)[:, tf.newaxis]
        j = tf.range(sequence_length)
        mask = tf.cast(i >= j, dtype="int32")
        mask = tf.reshape(mask, (1, input_shape[1], input_shape[1]))
        mult = tf.concat(
            [tf.expand_dims(batch_size, -1),
             tf.constant([1, 1], dtype=tf.int32)], axis=0)
        return tf.tile(mask, mult)

    def call(self, inputs, encoder_outputs, mask=None):
        causal_mask = self.get_causal_attention_mask(inputs)
        if mask is not None:
            padding_mask = tf.cast(
                mask[:, tf.newaxis, :], dtype="int32")
            padding_mask = tf.minimum(padding_mask, causal_mask)
        else:
            padding_mask = mask
        attention_output_1 = self.attention_1(
            query=inputs,
            value=inputs,
            key=inputs,
            attention_mask=causal_mask)
        attention_output_1 = self.layernorm_1(inputs + attention_output_1)
        attention_output_2 = self.attention_2(
            query=attention_output_1,
            value=encoder_outputs,
            key=encoder_outputs,
            attention_mask=padding_mask,
        )
        attention_output_2 = self.layernorm_2(
            attention_output_1 + attention_output_2)
        proj_output = self.dense_proj(attention_output_2)
        return self.layernorm_3(attention_output_2 + proj_output)

In [9]:
embed_dim = 256
latent_dim = 2048
num_heads = 2

inputs = keras.Input(shape=(None,), dtype="int64")
x = PositionalEmbedding(sequence_length, vocab_size, embed_dim)(inputs)
x = TransformerDecoder(embed_dim, latent_dim, num_heads)(x, x)
outputs = layers.Dense(vocab_size, activation="softmax")(x)
model = keras.Model(inputs, outputs)
model.compile(loss="sparse_categorical_crossentropy", optimizer="rmsprop")

In [10]:
tokens_index = dict(enumerate(text_vectorization.get_vocabulary()))

def sample_next(predictions, temperature=1.0):
    predictions = np.asarray(predictions).astype("float64")
    predictions = np.log(predictions) / temperature
    exp_preds = np.exp(predictions)
    predictions = exp_preds / np.sum(exp_preds)
    probas = np.random.multinomial(1, predictions, 1)
    return np.argmax(probas)

class TextGenerator(keras.callbacks.Callback):
    def __init__(self,
                 prompt,
                 generate_length,
                 model_input_length,
                 temperatures=(1.,),
                 print_freq=1):
        self.prompt = prompt
        self.generate_length = generate_length
        self.model_input_length = model_input_length
        self.temperatures = temperatures
        self.print_freq = print_freq
        vectorized_prompt = text_vectorization([prompt])[0].numpy()
        self.prompt_length = np.nonzero(vectorized_prompt == 0)[0][0]

    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % self.print_freq != 0:
            return
        for temperature in self.temperatures:
            print("== Generating with temperature", temperature)
            sentence = self.prompt
            for i in range(self.generate_length):
                tokenized_sentence = text_vectorization([sentence])
                predictions = self.model(tokenized_sentence)
                next_token = sample_next(
                    predictions[0, self.prompt_length - 1 + i, :]
                )
                sampled_token = tokens_index[next_token]
                sentence += " " + sampled_token
            print(sentence)

prompt = "This movie"
text_gen_callback = TextGenerator(
    prompt,
    generate_length=50,
    model_input_length=sequence_length,
    temperatures=(0.2, 0.5, 0.7, 1., 1.5))

In [11]:
#| output: false

'''model.fit(lm_dataset, epochs=2, callbacks=[text_gen_callback])'''

# 20 minuten per epoch op mijn laptop met CPU
# 3 minuten per epoch op colab met GPU

'model.fit(lm_dataset, epochs=2, callbacks=[text_gen_callback])'

In [12]:
model.load_weights('/Users/Ik/Desktop/2023-2024/AAI/MenT/Notebooks/models/colab/Gewichten50.weights.h5', by_name=True)

### Gegenereerde zinnen

In [13]:
for temperature in (0.2, 0.5, 0.7, 1., 1.5):
  print("== \033[1mGenerating with temperature", temperature,"\033[0m")
  prompt = "This movie"
  sentence = prompt
  vectorized_prompt = text_vectorization([prompt])[0].numpy()
  prompt_length = np.nonzero(vectorized_prompt == 0)[0][0]
  for i in range(50):
    tokenized_sentence = text_vectorization([sentence])
    predictions = model(tokenized_sentence)
    next_token = sample_next(
        predictions[0, prompt_length - 1 + i, :]
        )
    sampled_token = tokens_index[next_token]
    sentence += " " + sampled_token
  print(sentence)

== Generating with temperature 0.2 
This movie third hilarity cents youngster zu revelation wholeheartedly utmost maid suggesting concerts exclusive foxy chock progressively survived 1968 might titanic anastasia dates fashions scoop lions families chaplin display representation camps lynchs immigrants incessant unto disgust bring chavez practicing murderers referred macarthur elton notwithstanding consistency cher borrow rudy picking older 1993 coin
== Generating with temperature 0.5 
This movie masterpiece matters noting digress playwright archives pinned icon loyal coke dodgy moviei requiem florinda social boundaries dani repetition sydow romances sank bestselling conscience bittersweet degree wrap lambert son vic seeks norway dispute village expose btw patriotism driving lori convoluted jeep comprised relax dreamworks indicates art muppet has innocent ending competing
== Generating with temperature 0.7 
This movie contributing right marginal hiding orientation patton geographic pati

### Bespreek resultaten
- Een lage temperatuur zou voor meer "saaie" en herhalende teksten moeten zorgen.
- Een hogere temperatuur zorgt voor "interessantere", verassendere en "creatievere" teksten.
- Welke temperatuur is geschikter voor dit probleem?

Probeer altijd verschillende temperaturen om de juiste balans tussen structuur en willekeur te vinden.

### Hoe gebruiken wij taal?
- Wij gebruiken taal om
    - te informeren,
    - mensen te laten lachen, huilen, nadenken, op hun gemak te stellen,
    - te onderwijzen,
    - gedachten te ordenen.
    - ...
- Dit alles doet een taalmodel niet.

### Hoe gebruikt een taalmodel onze taal?
- Een taalmodel informeert niet.
    - Het wil niet informeren en er is niemand om te informeren
- Een taalmodel probeert niet mensen te beinvloeden of te onderwijzen
    - Het heeft geen voornemen of wil
- Een taalmodel probeert geen gedachten te ordenen

<br>

- Een taalmodel gebruikt de statistische structuur van de teksten die het gevoerd krijgt.<br>
Zie ook paragraaf 15.7.5 uit [PML1] voor discussies over large language models en wat ze wel en niet kunnen "begrijpen".